# D-Mart Retail Forecasting — Exploratory Analysis
This notebook reproduces the EDA, dataset characterization, and key figures from the paper.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'font.family': 'serif', 'figure.dpi': 120})

## 1. Load and inspect the raw D-Mart dataset

In [ ]:
# Load raw data
df = pd.read_csv('../data/new_database.csv', parse_dates=['Date'])
print(f'Shape: {df.shape}')
print(f'Date range: {df["Date"].min()} to {df["Date"].max()}')
print(f'Products: {df["Product_ID"].nunique()} SKUs')
print(f'Categories: {df["Product_Category"].unique()}')
print(f'Columns: {list(df.columns)}')
df.head()

## 2. Category-level daily aggregation

In [ ]:
cats = ['Food', 'Electronics', 'Clothing', 'Furniture']
cat_series = {}

for cat in cats:
    s = df[df['Product_Category']==cat].groupby('Date')['Sales'].sum()
    cat_series[cat] = s
    cv = s.std() / s.mean()
    adf_p = adfuller(s)[1]
    ac1 = s.autocorr(1)
    print(f'{cat:12}: n={len(s)}, mean={s.mean():.0f}, std={s.std():.0f}, '
          f'CV={cv:.3f}, AC(1)={ac1:.3f}, ADF p={adf_p:.4f}')

## 3. Time series plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
colors = ['#E53935', '#1E88E5', '#43A047', '#FB8C00']

for ax, (cat, color) in zip(axes.flatten(), zip(cats, colors)):
    s = cat_series[cat]
    ax.plot(s.index, s.values, color=color, lw=1.5)
    ax.set_title(f'{cat} — Daily Sales (Category Aggregate)', fontweight='bold')
    ax.set_ylabel('Sales')
    ax.set_xlabel('Date')

plt.suptitle('D-Mart Category-Level Daily Sales (Jan–Jun 2023)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

## 4. ACF / PACF analysis — confirming near-white-noise

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 14))

for i, cat in enumerate(cats):
    s = cat_series[cat]
    plot_acf(s, lags=30, ax=axes[i][0], title=f'{cat} — ACF')
    plot_pacf(s, lags=30, ax=axes[i][1], title=f'{cat} — PACF')

plt.suptitle('ACF/PACF: Near-zero autocorrelation confirms low-SNR regime',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()
print('Key finding: All series show near-zero autocorrelation at all lags.')
print('This explains why ARIMA(0,0,0) is AIC-optimal on Food and Clothing.')

## 5. Exogenous variable correlations

In [ ]:
total = df.groupby('Date').agg(
    Sales=('Sales','sum'),
    Promotion=('Promotion','mean'),
    Price=('Price','mean'),
    External=('External_Factors','mean')
).reset_index()

print('Correlations with total daily sales:')
print(total[['Sales','Promotion','Price','External']].corr()['Sales'])
print()
print('Key finding: Promotion r=-0.09, Price r=0.02.')
print('Individual promotional effects are washed out by aggregation across 800 SKUs.')
print('This explains why ARIMAX underperforms plain ARIMA at category level.')

## 6. SKU-level CV distribution

In [ ]:
np.random.seed(42)
sku_cvs = []
sample_prods = np.random.choice(df['Product_ID'].unique(), 100, replace=False)

for prod in sample_prods:
    s = df[df['Product_ID']==prod]['Sales'].values
    if len(s) > 30 and s.mean() > 0:
        sku_cvs.append(s.std() / s.mean())

cat_cvs = [s.std()/s.mean() for s in cat_series.values()]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(sku_cvs, bins=20, color='#E53935', alpha=0.7, label=f'SKU-level (n=100, mean={np.mean(sku_cvs):.3f})')
ax.axvline(np.mean(cat_cvs), color='#2196F3', lw=3,
           label=f'Category-level mean CV = {np.mean(cat_cvs):.3f}')
ax.axvline(0.03, color='#FF9800', lw=2, ls='--', label='CV=0.03 threshold')
ax.set_xlabel('Coefficient of Variation (CV)')
ax.set_ylabel('Count')
ax.set_title('SKU-Level vs Category-Level CV Distribution\n'
             f'Aggregation reduces CV by ~{np.mean(sku_cvs)/np.mean(cat_cvs):.0f}×',
             fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 7. Load and display benchmark results

In [ ]:
import json
from pathlib import Path

results = {}
for fname in Path('../results').glob('*_results.json'):
    with open(fname) as f:
        results[fname.stem.replace('_results','')] = json.load(f)

# Pretty-print RMSE table
datasets = ['Real_Food','Real_Electronics','Real_Clothing','Real_Furniture','walmart','m5']
models   = ['ARIMA','SARIMA','Prophet','XGBoost','LSTM','Hybrid']

rows = []
for m in models:
    row = {'Model': m}
    for d in datasets:
        row[d] = results.get(d, {}).get('metrics', {}).get(m, {}).get('rmse', float('nan'))
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('Model')
print('RMSE Summary (bold = best per column):')
summary_df.round(2)